# Análisis completo — Optimizador multiagente de flota anchovetera

Este notebook ejecuta el proyecto, muestra tablas, mapas, costos y simulación.

**Regla de evidencia:** si existe `data/raw/MapProbabilidad_adulto.csv`, se usa como `CLIENT_INPUT`. Si no existe, se ejecuta el modo demo y la probabilidad queda marcada como `SIMULATED_DEMO`. Las rutas y movimientos de barcos son siempre resultados simulados del modelo.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import Image, display, HTML

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'outputs'
CLIENT = ROOT / 'data' / 'raw' / 'MapProbabilidad_adulto.csv'
print('ROOT:', ROOT.resolve())
print('Archivo cliente:', CLIENT.exists())

## 1. Ejecutar pipeline

El precio de combustible de S/ 5.00/L usado aquí es un **parámetro de escenario**, no un precio oficial observado. Cámbialo si cuentas con un valor real.

In [ ]:
cmd = [sys.executable, str(ROOT/'scripts'/'run_pipeline.py'), '--sernanp', 'auto', '--precio-combustible', '5.0']
if not CLIENT.exists():
    cmd.append('--demo')
    print('No se encontró el CSV del proyecto: se ejecutará SIMULATED_DEMO.')
else:
    print('Se usará CLIENT_INPUT.')

subprocess.check_call(cmd)

## 2. Auditoría de procedencia

In [ ]:
auditoria = json.loads((OUT/'auditoria.json').read_text(encoding='utf-8'))
auditoria

## 3. Zonas candidatas

In [ ]:
zonas = pd.read_csv(OUT/'zonas_candidatas.csv')
display(zonas)

In [ ]:
display(Image(filename=str(OUT/'mapa_probabilidad_zonas.png')))

## 4. Comparación Greedy vs MILP

La comparación considera probabilidad de la zona, distancia total ida+retorno, combustible y costo económico de escenario.

In [ ]:
resumen = pd.read_csv(OUT/'resumen_metodos.csv')
display(resumen)

In [ ]:
display(Image(filename=str(OUT/'grafico_comparacion_metodos.png')))

## 5. Asignación Greedy

In [ ]:
greedy = pd.read_csv(OUT/'asignacion_greedy.csv')
display(greedy)

In [ ]:
display(Image(filename=str(OUT/'mapa_asignacion_greedy.png')))

## 6. Asignación MILP

In [ ]:
milp = pd.read_csv(OUT/'asignacion_milp.csv')
display(milp)

In [ ]:
display(Image(filename=str(OUT/'mapa_asignacion_milp.png')))

## 7. Costos por embarcación

In [ ]:
costos_barco = pd.read_csv(OUT/'costos_por_barco.csv')
cols = ['agent_id','port','zone_id','prob','distancia_total_nm','horas_total','combustible_total_l','costo_total_pen']
display(costos_barco[cols])

In [ ]:
display(Image(filename=str(OUT/'grafico_costos_por_barco.png')))

## 8. Costos y combustible por puerto

In [ ]:
costos_puerto = pd.read_csv(OUT/'costos_por_puerto.csv')
display(costos_puerto)

In [ ]:
display(Image(filename=str(OUT/'grafico_combustible_por_puerto.png')))

## 9. Ocupación de zonas y coordinación

In [ ]:
display(Image(filename=str(OUT/'grafico_ocupacion_zonas.png')))

## 10. Trade-off distancia vs probabilidad

In [ ]:
display(Image(filename=str(OUT/'grafico_distancia_vs_probabilidad.png')))

## 11. Simulación animada

La animación es una **simulación del movimiento planeado**. No representa trayectorias históricas observadas.

In [ ]:
gif = OUT/'simulacion_flota.gif'
display(Image(filename=str(gif)))

## 12. Reporte HTML completo

In [ ]:
print('Reporte:', (OUT/'reporte_resultados.html').resolve())
display(HTML('<b>Abre outputs/reporte_resultados.html en tu navegador para revisar todas las tablas y gráficos juntos.</b>'))

## 13. Lectura recomendada de resultados

1. Verifica primero `auditoria.json`.
2. Compara Greedy y MILP en probabilidad, distancia, combustible y costo.
3. Revisa si una zona concentra demasiadas embarcaciones.
4. Evalúa el gráfico distancia-probabilidad para detectar rutas caras con poca ganancia.
5. Usa la simulación para explicar la coordinación espacial.
6. No conviertas `Prob` en toneladas: para eso faltaría CPUE/biomasa/captura condicional.